
# Step-by-Step Estimation Notebook
**Dynamic Portfolio Choice with Budget-Embedded Financial Literacy (NFXP + EM)**

This notebook walks through the full estimation pipeline, step by step:

1. Load the panel and inspect key columns.
2. Build discretization grids and initialize the dynamic model.
3. Solve the dynamic program (NFXP) for one type and interpret the results.
4. Construct the per-id likelihood from model-implied P(x|state).
5. Run the EM algorithm with observable-dependent type priors.
6. Inspect saved estimates, responsibilities, policies, and CCPs.
7. Visualize policies and a few model objects.

> **Model summary:** CRRA utility, risky & safe assets, and *financial literacy costs enter the budget constraint*. If the risky share `x > 0`, you pay a fixed cost (toll) and an ad-valorem fee (tax), which reduce resources and thus both consumption and next-period assets.


In [6]:
# %pip -q install numpy pandas scipy matplotlib

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nfxp_model_v2 import NFXPFinLitConsBudgetAgeType, SpecType
from estimate_em_obs_v2 import (load_data, prep_grids, build_features, per_id_loglike_for_type, run_em_obs)


Note: you may need to restart the kernel to use updated packages.


## 1) Paths & dataset

In [7]:

CSV = "toy_panel_with_random_x.csv"  # provided dataset
OUT = "outputs_em_obs_nb"     # notebook-specific outputs
os.makedirs(OUT, exist_ok=True)

df = load_data(CSV)
df.head()


,id,t,age,a,y,x
0,0,0,49,22578.384369,18562.351115,0.986390
1,0,1,50,21927.092309,18448.435478,0.242381
2,0,2,51,21828.647582,18243.992208,0.702797
3,0,3,52,21654.901444,18939.917391,0.695454
4,0,4,53,21921.181994,18889.896102,0.009471



**Data expectations** (per row):

- `id`: individual identifier  
- `t`: time index within the individual's panel  
- `a`: assets entering period `t`  
- `y`: income (we use its mean for `ȳ` in this first pass)  
- `x`: observed risky share in \([0,1]\) (we discretize to a grid for the likelihood)  
- `age`: age or an age index  


## 2) Build grids and initialize the dynamic model

In [9]:

a_grid, age_grid, x_grid, s_grid, y_bar = prep_grids(df, n_a=24) # n is the number of points in the asset grid
print("Asset grid range:", a_grid[0], "…", a_grid[-1], " (N =", len(a_grid), ")")
print("Age grid length:", len(age_grid))
print("Action grids:", x_grid, s_grid)
print("Income mean y_bar:", y_bar)

model = NFXPFinLitConsBudgetAgeType(a_grid, age_grid, y_bar, x_grid, s_grid, gh_order=5)


Asset grid range: 13871.664729430238 … 42718.45478954852  (N = 24 )
Age grid length: 25
Action grids: [0.   0.25 0.5  0.75 1.  ] [0.6  0.8  0.95]
Income mean y_bar: 22853.649914078767


## 3) Solve the DP for a single type (demonstration)

- Safe gross return $R_f > 0.$

- Risky gross return $R = \exp(\mu_{\ln R} + \sigma_{\ln R} Z), \quad Z \sim \mathcal{N}(0, 1) \text{ i.i.d. across periods.}$

- Expectation over $R$ is computed by Gauss–Hermite quadrature.



In [10]:

# Choose a simple parameterization for demonstration
spec_demo = SpecType(
    beta=0.95, sigma=2.0,
    R_f=1.02, mu_lnR=np.log(1.06), sigma_lnR=0.20,
    gamma0=1.0, gamma1=-0.01,   # fixed cost ~ exp(1.0 - 0.01*age)
    delta0=-1.0, delta1=0.01    # tau ~ 0.9*sigmoid(-1.0 + 0.01*age)
)

V_demo, CCP_demo, polx_demo, pols_demo = model.value_iteration_one_type(spec_demo)
print("V_demo shape:", V_demo.shape, " CCP_demo shape:", CCP_demo.shape)
print("Policy x* sample (first 5 states along age=0):", polx_demo[:5, 0])
print("Policy s* sample (first 5 states along age=0):", pols_demo[:5, 0])


V_demo shape: (24, 25)  CCP_demo shape: (24, 25, 5, 3)
Policy x* sample (first 5 states along age=0): [0. 0. 0. 0. 0.]
Policy s* sample (first 5 states along age=0): [0.6 0.8 0.8 0.8 0.8]


## 4) Likelihood construction (per type)

In [ ]:

# For the demo, compute the per-id log-likelihood under this type
ell_demo, id_order_demo = per_id_loglike_for_type(df, model, CCP_demo)
print("Per-id log-likelihood (demo type) — first 10:")
pd.Series(ell_demo[:10], index=id_order_demo[:10])


## 5) Run EM with observable-dependent mixture weights

In [ ]:

# We keep the EM short for demonstration; increase em_iters/nm_steps for a full run.
res = run_em_obs(CSV, outdir=OUT, n_a=24, gh_order=5, em_iters=3, nm_steps=15, seed=1)
res


## 6) Inspect saved outputs

In [ ]:

est = pd.read_csv(os.path.join(OUT, "estimates.csv"))
resp = pd.read_csv(os.path.join(OUT, "responsibilities.csv"))
print("Estimates:"); display(est)
print("Responsibilities (head):"); display(resp.head())


## 7) Visualize policies x* and s* over (a, age)

In [ ]:

# Load policies
pol1 = pd.read_csv(os.path.join(OUT, "policy_type1.csv"))
pol2 = pd.read_csv(os.path.join(OUT, "policy_type2.csv"))

# Pivot into matrices shaped (NA, NG). We know grid lengths from the model above.
NA = len(a_grid); NG = len(age_grid)
X1 = pol1['x_star'].to_numpy().reshape(NA, NG, order='C')
S1 = pol1['s_star'].to_numpy().reshape(NA, NG, order='C')
X2 = pol2['x_star'].to_numpy().reshape(NA, NG, order='C')
S2 = pol2['s_star'].to_numpy().reshape(NA, NG, order='C')

# Plot x* heatmaps for type 1 and 2
plt.figure(figsize=(6,4))
plt.imshow(X1, aspect='auto', origin='lower')
plt.title("Type 1: x* over (asset index, age index)")
plt.xlabel("age index")
plt.ylabel("asset index")
plt.colorbar()
plt.show()

plt.figure(figsize=(6,4))
plt.imshow(X2, aspect='auto', origin='lower')
plt.title("Type 2: x* over (asset index, age index)")
plt.xlabel("age index")
plt.ylabel("asset index")
plt.colorbar()
plt.show()


## 8) CCP slice: P(x | a, age) at a selected state

In [ ]:

# Load CCP arrays and visualize P(x|state) for a middle asset and age index
CCP1 = np.load(os.path.join(OUT, "CCP_type1.npy"))
mid_ai = len(a_grid)//2
mid_gi = len(age_grid)//2

Px_type1 = CCP1[mid_ai, mid_gi, :, :].sum(axis=1)  # sum over savings
Px_type1 = Px_type1 / Px_type1.sum()

plt.figure(figsize=(6,3))
plt.bar(range(len(x_grid)), Px_type1)
plt.xticks(range(len(x_grid)), x_grid)
plt.title("Type 1: P(x | middle state)")
plt.xlabel("x grid")
plt.ylabel("probability")
plt.show()



## 9) What to try next
- **Increase EM iterations and NM steps** for convergence on your machine.
- **Richer cost functions:** edit `_kfix` and `_tau` in `nfxp_model.py` to depend on `ln(a)` and other observables.
- **More types:** extend EM from 2 to L types (the bookkeeping generalizes naturally).
- **Observed consumption:** if available, add a consumption-likelihood term to sharpen \((\beta, \sigma)\).
- **Income dynamics:** replace `ȳ` with a Markov income process and integrate income shocks in the DP.



## 10) Smoke tests (sanity checks)
We quickly verify:
- CCP rows (over actions) sum to ~1 for each state.
- Marginalized probabilities over `x` sum to ~1 along observed paths.
- Policies are on-grid and feasible.


In [ ]:

import numpy as np, os, pandas as pd

# Load CCPs and policies from earlier outputs
CCP1 = np.load(os.path.join(OUT, "CCP_type1.npy"))
CCP2 = np.load(os.path.join(OUT, "CCP_type2.npy"))
pol1 = pd.read_csv(os.path.join(OUT, "policy_type1.csv"))
pol2 = pd.read_csv(os.path.join(OUT, "policy_type2.csv"))

# 1) CCP normalization check
def check_ccp_norm(CCP, name):
    NA, NG, NX, NS = CCP.shape
    bad = 0
    for ai in range(0, NA, max(1, NA//6)):
        for gi in range(0, NG, max(1, NG//6)):
            s = CCP[ai, gi, :, :].sum()
            if not np.isfinite(s) or abs(s - 1.0) > 1e-6:
                bad += 1
    print(f"[{name}] CCP normalization issues at sampled points:", bad)

check_ccp_norm(CCP1, "Type 1")
check_ccp_norm(CCP2, "Type 2")

# 2) P(x|state) sums to 1 (using demo model already constructed above)
Px = CCP1[len(a_grid)//2, len(age_grid)//2, :, :].sum(axis=1)
print("Px sum (Type 1, mid state) =", float(Px.sum()))

# 3) Policy feasibility: values lie on provided grids
assert set(np.unique(pol1['x_star'])).issubset(set(x_grid)), "Type1 x* off-grid"
assert set(np.unique(pol2['x_star'])).issubset(set(x_grid)), "Type2 x* off-grid"
assert set(np.unique(pol1['s_star'])).issubset(set(s_grid)), "Type1 s* off-grid"
assert set(np.unique(pol2['s_star'])).issubset(set(s_grid)), "Type2 s* off-grid"
print("Policy feasibility checks passed.")



## 11) Optional: longer EM run (heavier)
If you want tighter convergence, increase iterations/steps below. This will take longer.


In [ ]:

# Increase EM/NM for a more converged solution (runtime heavier).
res_long = run_em_obs(CSV, outdir=OUT, n_a=32, gh_order=7, em_iters=8, nm_steps=50, seed=2)
res_long
